# 实验一（课堂演示）：AI 驱动的网络流量分类

**适用课程**：未来媒体互联网  
**演示时长**：约 10 分钟  
**运行环境**：Kaggle Notebook（CPU 即可）

## 演示目标

让学生看到：机器学习模型只需几行代码，就能自动识别"这段流量是 YouTube 还是 DNS"，而不需要人工看端口号。

## 课前准备

本 Notebook 使用预生成的流量特征数据（CSV 格式），无需上传 pcapng 文件。如需从原始 pcapng 提取特征，请参考完整版实验指导书。

In [ ]:
# 环境准备（Kaggle 已预装 scikit-learn，只需确认版本）
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

print("所有依赖已就绪 ✅")

In [ ]:
# 生成演示数据（模拟真实流量特征）
# 4 类流量：Web(0) / DNS(1) / Video(2) / Download(3)
# 特征：平均包长、包长方差、包间隔均值、包间隔方差、字节数、端口号

np.random.seed(42)
n_samples = 200

def generate_flow(label, n, pkt_mean, pkt_var, gap_mean, gap_var, port):
    return pd.DataFrame({
        'pkt_len_mean': np.random.normal(pkt_mean, pkt_mean*0.15, n),
        'pkt_len_var': np.random.normal(pkt_var, pkt_var*0.3, n),
        'gap_mean': np.random.normal(gap_mean, gap_mean*0.2, n),
        'gap_var': np.random.normal(gap_var, gap_var*0.3, n),
        'bytes': np.random.normal(pkt_mean * 20, pkt_mean * 5, n),
        'port': np.random.normal(port, 5, n).astype(int),
        'label': label
    })

df = pd.concat([
    generate_flow(0, n_samples, 800, 50000, 0.05, 0.02, 80),    # Web: 小包、80端口
    generate_flow(1, n_samples, 100, 500, 0.5, 0.1, 53),        # DNS: 极小包、53端口
    generate_flow(2, n_samples, 1400, 100000, 0.01, 0.005, 443), # Video: 大包、443端口
    generate_flow(3, n_samples, 1500, 50000, 0.001, 0.001, 21),  # Download: 满包、21端口
])

labels = ['Web', 'DNS', 'Video', 'Download']
print(f"生成 {len(df)} 条流量记录")
df.head()

In [ ]:
# 可视化：不同类型的流量在特征空间中的分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i, label in enumerate(labels):
    subset = df[df['label'] == i]
    axes[0].scatter(subset['pkt_len_mean'], subset['gap_mean'], 
                   label=label, alpha=0.6, s=20)

axes[0].set_xlabel('平均包长 (bytes)')
axes[0].set_ylabel('平均包间隔 (s)')
axes[0].set_title('流量特征空间分布')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 只用端口号分类 vs 用统计特征分类 —— 直观对比
X_port = df[['port']]
X_stats = df[['pkt_len_mean', 'pkt_len_var', 'gap_mean', 'gap_var', 'bytes']]
y = df['label']

X_train_p, X_test_p, y_train, y_test = train_test_split(X_port, y, test_size=0.3, random_state=42)
X_train_s, X_test_s, _, _ = train_test_split(X_stats, y, test_size=0.3, random_state=42)

knn_port = KNeighborsClassifier(n_neighbors=5).fit(X_train_p, y_train)
knn_stats = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)

acc_port = knn_port.score(X_test_p, y_test)
acc_stats = knn_stats.score(X_test_s, y_test)

axes[1].bar(['仅用端口号', '统计特征（5维）'], [acc_port, acc_stats], 
           color=['#e74c3c', '#2ecc71'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('分类准确率')
axes[1].set_title('特征选择对分类结果的影响')
for i, v in enumerate([acc_port, acc_stats]):
    axes[1].text(i, v + 0.02, f'{v:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()
print(f"\n仅用端口号：{acc_port:.1%}  |  统计特征：{acc_stats:.1%}")
print("为什么统计特征远优于端口号？因为同一端口可能承载多种流量类型（如 443 可能是 Web 也可能是视频）")

In [ ]:
# 混淆矩阵：看看哪两类最容易混淆
y_pred = knn_stats.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.xlabel('预测类别')
plt.ylabel('真实类别')
plt.title('混淆矩阵：KNN + 统计特征')
plt.show()

print("\n课堂互动问题：")
print("1. 对角线上的数字代表什么？")
print("2. 哪两个类别最容易混淆？为什么？")
print("3. 如果让你加一个新特征来改善，你会加什么？")

## 课堂演示流程（10 分钟）

1. **0-2 min**：运行环境检查 + 数据生成，展示流量特征表格
2. **2-5 min**：运行特征空间散点图，让学生观察"不同类型流量确实聚在不同的位置"
3. **5-8 min**：对比"仅端口号"和"统计特征"的准确率柱状图，引出第一个结论
4. **8-10 min**：展示混淆矩阵，现场提问

## 课后延伸

学生可在完整版实验指导书中找到从 pcapng 提取特征的完整代码，
并对比 KNN、决策树、随机森林三种分类器的效果。